In [1]:
import numpy as np
import torch
import scipy.io as sio
from sklearn.decomposition import SparseCoder, MiniBatchDictionaryLearning
from torch import nn
from DL import DL
import time 
import matplotlib.pyplot as plt

In [2]:
def calculate_snr(clean_data, denoised_data):
    norm_clean = np.linalg.norm(clean_data)
    norm_diff = np.linalg.norm(clean_data - denoised_data)
    snr = 20 * np.log10(norm_clean / norm_diff)
    return snr

In [3]:
def patch3d(A, l1=4, l2=4, l3=4, s1=2, s2=2, s3=2):
    """
    Optimized patch3d function using stride tricks to improve efficiency
    """
    # Pad the array if necessary to ensure dimensions are divisible by patch sizes
    pad1 = (l1 - A.shape[0] % s1) % s1
    pad2 = (l2 - A.shape[1] % s2) % s2
    pad3 = (l3 - A.shape[2] % s3) % s3
    A_padded = np.pad(A, ((0, pad1), (0, pad2), (0, pad3)), mode='constant')

    # Get new dimensions
    n1, n2, n3 = A_padded.shape
    n1_patches = (n1 - l1) // s1 + 1
    n2_patches = (n2 - l2) // s2 + 1
    n3_patches = (n3 - l3) // s3 + 1

    # Generate the sliding window view
    shape = (n1_patches, n2_patches, n3_patches, l1, l2, l3)
    strides = (s1 * A_padded.strides[0], s2 * A_padded.strides[1], s3 * A_padded.strides[2]) + A_padded.strides
    patches = np.lib.stride_tricks.as_strided(A_padded, shape=shape, strides=strides)

    # Reshape patches to (num_patches, patch_size) format
    patches = patches.reshape(-1, l1 * l2 * l3)
    return patches


def patch3d_inv(X, n1, n2, n3, l1=4, l2=4, l3=4, s1=2, s2=2, s3=2):
    """
    Reconstruct 3D data from 1D patches with optimized inverse patching.

    INPUT
    X: Patches in (num_patches, patch_size) format
    n1, n2, n3: Original 3D data dimensions
    l1, l2, l3: Patch sizes along each dimension
    s1, s2, s3: Shifts between patches along each dimension

    OUTPUT
    A: Reconstructed 3D data
    """

    # Initialize the padded 3D array and mask
    pad1 = (l1 - n1 % s1) % s1
    pad2 = (l2 - n2 % s2) % s2
    pad3 = (l3 - n3 % s3) % s3
    A = np.zeros((n1 + pad1, n2 + pad2, n3 + pad3))
    mask = np.zeros_like(A)

    # Reshape 1D patches to the 3D patch size for easier handling
    X = X.reshape(-1, l1, l2, l3)

    # Calculate the number of patches along each dimension
    n1_patches = (A.shape[0] - l1) // s1 + 1
    n2_patches = (A.shape[1] - l2) // s2 + 1
    n3_patches = (A.shape[2] - l3) // s3 + 1

    # Iterate over each patch and place it into the appropriate location
    idx = 0
    for i in range(n1_patches):
        for j in range(n2_patches):
            for k in range(n3_patches):
                # Place the current patch in the reconstruction array and update mask
                A[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += X[idx]
                mask[i * s1:i * s1 + l1, j * s2:j * s2 + l2, k * s3:k * s3 + l3] += 1
                idx += 1

    # Avoid division by zero by replacing zeros with ones in the mask before division
    mask[mask == 0] = 1
    A /= mask

    # Trim padding to match the original dimensions
    return A[:n1, :n2, :n3]

In [4]:
# 1. 加载 3D 数据
data = sio.loadmat('Syn3d.mat')
Dc = data['DataClean'].astype(np.float32)
Dn = data['DataNoisy'].astype(np.float32)
[n1, n2, n3] = Dn.shape

snr_before = calculate_snr(Dc, Dn)

In [5]:
nz = Dn.shape[0]  # 对应MATLAB：n1 = size(Dn, 1)
nx = Dn.shape[1]  # 对应MATLAB：n2 = size(Dn, 2)
ny = Dn.shape[2]  # 对应MATLAB：n3 = size(Dn, 3)

In [6]:
# --- 2. 设置 3D 分块参数与设备 ---
w1, w2, w3 = 15, 15, 15
s1, s2, s3 = 2, 2, 2
input_size = w1 * w2 * w3

# === 您要求的 CUDA 设置代码 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # 打印当前使用的设备
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
# ===========================

# 初始化网络并移动到 GPU
net = DL(input_size).to(device)
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

niter = 5
epochs_per_iter = 10
alpha_list = (niter - np.arange(niter)) / (niter - 1)
d1 = Dn.copy()

In [8]:
total_start_time = time.time()
all_loss_history_3d = []

for it in range(niter):
    iter_start_time = time.time()
    print(f"\n===== 3D DDIUL 迭代 {it + 1}/{niter} =====")
    
    # --- 1. 3D 分块 (CPU操作) ---
    # Numpy 操作，只能在 CPU 进行
    X = patch3d(d1, w1, w2, w3, s1, s2, s3) 
    X = X.astype(np.float32)
    
    # 先转为 Tensor，暂时放在 CPU，筛选后再放入 GPU
    Pdata = torch.from_numpy(X)
    print(f"-> 分块后的总补丁数 (Total Patches): {X.shape[0]}")

    # --- 2. REBPS 筛选 (CPU操作 - 瓶颈) ---
    # 【注意】这是 Scikit-learn 代码，无法使用 CUDA 加速，速度取决于 CPU 核心数
    # 如果这里卡住太久，可以尝试减少 n_components 或 transform_n_nonzero_coefs
    print("-> 正在进行字典学习筛选 (Running Scikit-learn on CPU)...")
    dl_learner = MiniBatchDictionaryLearning(n_components=512, max_iter=20, n_jobs=-1) # n_jobs=-1 使用所有 CPU 核心
    dictionary = dl_learner.fit(X).components_
    coder = SparseCoder(dictionary=dictionary, transform_algorithm='omp', transform_n_nonzero_coefs=5, n_jobs=-1)
    
    # 计算误差用于筛选
    # transform 也是 CPU 操作
    coded_X = coder.transform(X)
    reconstructed_X = np.dot(coded_X, dictionary)
    error = np.linalg.norm(X - reconstructed_X, axis=1)
    
    selected_idx = np.argsort(error)[-int(len(X) * 0.7):]
    
    # 【关键】将筛选后的数据移动到 CUDA (GPU)
    train_tensor = Pdata[selected_idx].unsqueeze(1).to(device)
    print(f"-> REBPS 筛选完成，训练样本数: {len(selected_idx)} (已加载至 {device})")
    
    # --- 3. 神经网络训练 (GPU 加速) ---
    net = DL(w1*w2*w3).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    
    iter_loss = []
    best_loss = float('inf')
    patience_counter = 0
    patience = 3 
    
    net.train()
    # DataLoader 在 CPU 上组织 batch，然后送入 GPU
    loader = torch.utils.data.DataLoader(train_tensor, batch_size=128, shuffle=True)
    
    for epoch in range(epochs_per_iter):
        epoch_loss = 0.0
        for batch in loader:
            # 数据已经在 GPU 上 (train_tensor 定义时已 .to(device))
            # 但为了保险，通常在该处再次确认，或者 train_tensor如果在CPU这里加 .to(device)
            # 您的代码 train_tensor 已经在 device 上，这里不需要再次 .to(device) 但加了也不报错
            batch = batch.to(device) 
            
            optimizer.zero_grad()
            output = net(batch) # 模型在 GPU
            loss = loss_fn(output, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        iter_loss.append(avg_loss)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
        else:
            patience_counter += 1
            
        if (epoch + 1) % 2 == 0:
            print(f"Epoch [{epoch+1}/{epochs_per_iter}], Loss: {avg_loss:.6f}")
            
        if patience_counter >= patience:
            print(f"--> Early Stopping at epoch {epoch+1}")
            break
            
    all_loss_history_3d.append(iter_loss)
    
    # --- 4. 3D 数据重构 (GPU 加速) ---
    net.eval()
    with torch.no_grad():
        # 将所有补丁移动到 GPU 进行预测
        all_p = Pdata.unsqueeze(1).to(device)
        
        # 批量预测防止显存溢出
        batch_size_infer = 64 # 如果显存够大，可以调大这个数加快速度
        recon_list = []
        for i in range(0, len(all_p), batch_size_infer):
            batch_input = all_p[i:i+batch_size_infer]
            batch_output = net(batch_input)
            recon_list.append(batch_output)
            
        recon_X_torch = torch.cat(recon_list)
        recon_X = recon_X_torch.cpu().squeeze().numpy() # 传回 CPU 给 numpy 处理
        
    d2 = patch3d_inv(recon_X, n1, n2, n3, w1, w2, w3, s1, s2, s3)
    d1 = alpha_list[it] * Dn + (1 - alpha_list[it]) * d2
    
    # --- 5. 显存清理 ---
    del train_tensor, all_p, recon_X_torch # 删除 GPU 变量
    torch.cuda.empty_cache()
    print(f"本轮迭代耗时: {time.time() - iter_start_time:.2f} 秒")

print(f"\n[完成] 3D总耗时: {(time.time() - total_start_time)/60:.2f} 分钟")


===== 3D DDIUL 迭代 1/5 =====
-> 分块后的总补丁数 (Total Patches): 36288
-> REBPS 筛选后的训练补丁数: 25401
Epoch [2/10], Loss: 0.049656
Epoch [4/10], Loss: 0.044773
Epoch [6/10], Loss: 0.043254
Epoch [8/10], Loss: 0.042450
Epoch [10/10], Loss: 0.041948
本轮迭代耗时: 457.99 秒

===== 3D DDIUL 迭代 2/5 =====
-> 分块后的总补丁数 (Total Patches): 36288
-> REBPS 筛选后的训练补丁数: 25401
Epoch [2/10], Loss: 0.075872
Epoch [4/10], Loss: 0.068830
Epoch [6/10], Loss: 0.066716
Epoch [8/10], Loss: 0.065615
Epoch [10/10], Loss: 0.064933
本轮迭代耗时: 374.19 秒

===== 3D DDIUL 迭代 3/5 =====
-> 分块后的总补丁数 (Total Patches): 36288
-> REBPS 筛选后的训练补丁数: 25401
Epoch [2/10], Loss: 0.050362
Epoch [4/10], Loss: 0.045089
Epoch [6/10], Loss: 0.043398
Epoch [8/10], Loss: 0.042541
Epoch [10/10], Loss: 0.042024
本轮迭代耗时: 365.79 秒

===== 3D DDIUL 迭代 4/5 =====
-> 分块后的总补丁数 (Total Patches): 36288
-> REBPS 筛选后的训练补丁数: 25401
Epoch [2/10], Loss: 0.031392
Epoch [4/10], Loss: 0.026686
Epoch [6/10], Loss: 0.025306
Epoch [8/10], Loss: 0.024624
Epoch [10/10], Loss: 0.024221
本轮迭代耗

In [ ]:
# --- 损失曲线可视化 ---
plt.figure(figsize=(10, 5))
for i, losses in enumerate(all_loss_history_3d):
    plt.plot(losses, label=f'Iteration {i+1}')
plt.title('DDIUL 3D Training Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.yscale('log')
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

In [ ]:
# 4. 评估与保存
snr_after = calculate_snr(Dc, d1)
print(f"\n3D 去噪前 SNR: {snr_before:.2f} dB")
print(f"3D 去噪后 SNR: {snr_after:.2f} dB")
print(f"SNR 提升: {snr_after - snr_before:.2f} dB")

sio.savemat(r"output_syn3d.mat", {'D_denoised': d1})

In [9]:

#15*15*15 2*2*2
sio.savemat(r"output_syn3d.mat", {'D_denoised': d1})


3D 去噪前 SNR: -2.17 dB
3D 去噪后 SNR: 8.32 dB
SNR 提升: 10.49 dB


In [1]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # 清理进程间显存缓存，进一步减少碎片

NameError: name 'torch' is not defined